In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive_output, HBox, VBox, Layout, fixed

from libraryFunctions import *
from auxillaryFunctions import *

build_dir,configType = "build", "Release" #"Debug" Release ##NOTE: make sure your are using correct version
moduleDir = os.path.join(build_dir, configType)
if moduleDir not in sys.path:
    sys.path.append(moduleDir)

import cathalTest

import ipywidgets as widgets
from ipywidgets import interactive_output, VBox, HBox, Label, Layout

simOutputData = None

def plotSimResult(simData, **params): #gateValues
    gateIndMap = {key:i for i,key in enumerate(['vGate1','vGate2','CaGate','spikeGate'])}
    conductances,gateVals = {},{}

    for key,val in params.items():
        if "vg_" in key:
            gateVals[key] = val
        elif "cond_" in key:
            conductances[key] = val
        elif "TS_" in key:
            TS = val
        else:
            raise KeyError(f"Unknown key {key} in param dict")

    for channel,val in conductances.items():
        channel = channel.split("_")[1]
        ind = simData['indexMaps']['neuronToChannelMap']['test'][1][0][channel]
        simData['conductance'][ind] = val

    for gateData,val in gateVals.items():
        _,channel,gate,prop = gateData.split("_")
        channelInd = simData['indexMaps']['neuronToChannelMap']['test'][1][0][channel]
        gateInd = gateIndMap[gate]
        simData[prop][channelInd][gateInd] = val
        #rest of the logic later

    runTime = 80
    stimInfoSpikes,stimInfoCurrents = {},{}

    # stimBiasCurrent(stimInfoCurrents,[0],-40,100,103,TS)
    # simData['indexMaps']['neuronToChannelMap][neuron population name][neuron number][segment number][channel]
    syns = [simData['indexMaps']['neuronToChannelMap']['test'][1][0]['AMPA']]
    stimSynapse(stimInfoSpikes,syns,30,30+1*TS,TS)
    # print('stimInfoSpikes',stimInfoSpikes)

    simDataCpp = deepcopy(simData)
    simDataCpp["debugRecording"] = np.zeros_like(simDataCpp["current"])
    dataCpp = solveSystemCpp(simDataCpp,TS,runTime,samplingInterval=1,stimInfoSpikes=stimInfoSpikes,stimInfoCurrents=stimInfoCurrents,
                            recordingKeys =["spikerOutput","voltages",'x'],singleThread=True)
    global simOutputData
    simOutputData = dataCpp 

    # dataCpp = solveSystemStandard(simDataCpp,TS,runTime,stimInfoSpikes,stimInfoCurrents,vectorized=True,STRANGSPLIT=False)


    AMPAsynIndex = simData['indexMaps']['neuronToChannelMap']['test'][1][0]['AMPA']
    spikeGate = dataCpp['x'][:,AMPAsynIndex,3] #index 3 is spike, 2 Ca, 1,0 are voltage
    
    V = dataCpp["voltages"]
    tv = dataCpp["time"]
    fig,ax = plt.subplots(2,1,figsize=(7,4),sharex=True,dpi=120)
    # ax = np.atleast_1d(ax)
    ax[0].plot(tv,V,label = simData["neuronName"])
    ax[0].set_title("Voltage")
    ax[0].set_ylabel("Vm (mV)")
    ax[0].legend()  
    ax[1].plot(tv,spikeGate,label='AMPAsyn spike gate')
    ax[1].legend()
    ax[1].set_xlabel('Time (ms)')


# simData = allParams['simData']
# condSliderList = []
# for neurPop, popData in simData['indexMaps']['neuronToChannelMap'].items():
#     for neurInd, segData in popData.items():
#         for segInd, channelDict in segData.items():
#             for channelName, channelInd in channelDict.items():
#                 c = simData['conductance'][channelInd]
#                 cSlider = widgets.FloatSlider(value=c,  min=0.0,  max=10*c, step=0.1*c,  description=f'',continuous_update=False) #turn continuous update off to stop flashing
#                 sliderName = f"cond_{neurInd}_{channelName}"
#                 allParams[sliderName] = cSlider 

#                 label = widgets.Label(value=f"{sliderName}:", layout=Layout(width='80px'))
#                 pairedWidget = HBox([label, cSlider], layout=Layout(justify_content='space-between'))
#                 condSliderList.append(pairedWidget)


def genUI_SingleNeuronSim(allParams,gateParameters,TS):
    gateIndMap = {key:i for i,key in enumerate(['vGate1','vGate2','CaGate','spikeGate'])}
    condSliderList = []
    # for key,val in channels.items():
    simData = allParams['simData'].value
    channelIndMap = simData['indexMaps']['neuronToChannelMap']['test'][1][0]

    for channelName, dataInd in channelIndMap.items():
        c = simData['conductance'][dataInd]
        # c = val['conductance']
        cSlider = widgets.FloatSlider(value=c,  min=0.0,  max=10*c, step=0.1*c,  description=f'',continuous_update=False) #turn continuous update off to stop flashing
        sliderName = f"cond_{channelName}"
        allParams[sliderName] = cSlider 

        label = widgets.Label(value=f"{sliderName}:", layout=Layout(width='80px'))
        pairedWidget = HBox([label, cSlider], layout=Layout(justify_content='space-between'))
        condSliderList.append(pairedWidget)

    gateSliderList = []
    for keys in gateParameters:
        channel,gate,prop, valRange = keys
        sliderName = f"vg_{channel}_{gate}_{prop}"
        channelProperty = simData[prop]
        assert len(channelProperty.shape) == 2, f'{prop} is not an appropriate gate property as it has a shape {channelProperty.shape}'
        c = channelProperty[channelIndMap[channel]][gateIndMap[gate]]
        # c = channels[channel]['gates'][gate][prop]
        gSlider = widgets.FloatSlider(value=c,  min=valRange[0],  max=valRange[1], step=(valRange[1]-valRange[0])/50,  description='',continuous_update=False)
        allParams[sliderName] = gSlider
        
        label = widgets.Label(value=f"{sliderName}:", layout=Layout(width='250px'))
        pairedWidget = HBox([label, gSlider], layout=Layout(justify_content='space-between'))
        gateSliderList.append(pairedWidget)


    # sliderName = 'TS_slider'
    # TSslider = widgets.FloatSlider(value=TS,  min=0.05,  max=0.3, step=0.01,  description='',continuous_update=False)
    # allParams[sliderName] = TSslider
    # label = widgets.Label(value=f"{sliderName}:", layout=Layout(width='150px'))
    # pairedWidget = HBox([label, TSslider], layout=Layout(justify_content='space-between'))
    # gateSliderList.append(pairedWidget)
    allParams['TS_slider'] = fixed(TS)


    # allSliders['simData'] = widgets.fixed(simData)
    widget_output = interactive_output(plotSimResult, allParams )

    title_label = Label(
        value="Interactive AIS Action Potential Modulator",
        style={'font_weight': 'bold'})

    ui_container = VBox([
        title_label,
        HBox([VBox(condSliderList, layout=Layout(justify_content='center')),VBox(gateSliderList, layout=Layout(justify_content='center'))]),
        widget_output],
        layout=Layout(align_items='center', padding='10px', border='1px solid #dcdfe5', border_radius='8px'))

    display(ui_container)

def solveSystemCpp(ac,TS,runTime,samplingInterval=2,stimInfoSpikes={},stimInfoCurrents = {}, 
                   recordingKeys = ["voltages","calciumVals","netCurrent","current"],singleThread=False):
    nIterations = int(runTime/TS)
    nSamples = int(np.ceil(nIterations/samplingInterval))
    spikeStimTimes,spikeStimNeurons = genSpikeStimArrays(stimInfoSpikes)
    ac["spikeStimTimes"] = spikeStimTimes
    ac["spikeStimNeurons"] = spikeStimNeurons

    biasStimTimes,biasStimSegments,biasStimValues = genBiasStimArrays(stimInfoCurrents)
    ac["biasStimTimes"] = biasStimTimes
    ac["biasStimSegments"] = biasStimSegments
    ac["biasStimValues"] = biasStimValues
    ac["performanceStats"] = np.zeros(8,dtype=np.float32)


    recordingData = {}
    for key in recordingKeys:
        shape = [nSamples] + list(ac[key].shape)
        recordingData[key] = np.zeros(shape,dtype=np.float32) #np.ascontiguousarray

    tv = np.linspace(0,runTime,nSamples)

    if singleThread:
        cathalTest.solveNeuralSystemSingleThread(ac,recordingData,TS,nIterations,samplingInterval)
    else:
        cathalTest.solveNeuralSystem(ac,recordingData,TS,nIterations,samplingInterval)
    
    #trackPerfOverTime(ac)
    recordingData["time"] = tv
    return recordingData

def examineChannelNormal(channel,ax=None,nSamples = 51):
    if ax is None:
        fig,ax = plt.subplots(3,3,figsize=(12,8), dpi=80)
        fig.suptitle(f"Channel: {channel['name']}, ion: {channel['ion']}")
    gateOpening = np.ones([nSamples])
    for i,(gate,gateData) in enumerate(channel["gates"].items()):
        if gate != "spikeGate":
            if gate == "CaGate":
                vr = np.linspace(0,1e-3,nSamples)
                xInf = examineGateNormal(gateData,vr,gate,[ax[0,i],ax[1,i]])
            else:
                valRange = np.linspace(-100,100,nSamples)
                xInf = examineGateNormal(gateData,valRange,gate,[ax[0,i],ax[1,i]])
                exponent = gateData["exponent"]
                gateOpening *= xInf**exponent

    ax[2,0].plot(valRange,gateOpening,c="orange",ls="--")
    plt.tight_layout()


    
def genGate(vHalf,vKi,tau0,tau1,th,tkInv,x0=0.0,exponent=1,minOpenness=0):
    return {"x":x0, "vHalf":vHalf,"vKi":vKi,"tau0":tau0,"tau1":tau1,"th":th,"tkInv":tkInv,"exponent":exponent,"minOpenness":minOpenness}

def genOpenGate():
    return genGate(-1000,1000,100,1000,0,0,x0=1)

def genSynapseGate(xt1,TS):
    xt0 = TS
    xt1 = xt1-xt0
    return genGate(0.5,vKi=20,tau0=xt0,tau1=xt1,th=0.5,tkInv=-20,exponent=2)

def sigmoid(z,h,kInv,minOp):
    return minOp +  (1.0-minOp)/(1.0 + np.exp((h-z)*kInv))

def examineGateNormal(gateData,valRange,gateName,ax=None):
    xInf = sigmoid(valRange,gateData["vHalf"],gateData["vKi"],gateData["minOpenness"]) 
    xTau = gateData["tau0"] + gateData["tau1"]*sigmoid(valRange,gateData["th"],gateData["tkInv"],0)
    gRange = [min(valRange),max(valRange)]

    if ax is None:
        fig,ax = plt.subplots(2,1,figsize=(8,6), dpi=100)
    
    ax[0].plot(valRange,xInf,c="b",ls="--")
    ax[0].axvline(gateData["vHalf"],color="g",linestyle="--")
    ax[0].set_title(f"{gateName} xInf")
    ax[0].set_xlim(gRange)
    #x[0].set_ylabel("Gate opening")
    ax[0].set_ylim(-0.01,1.01)

    ax[1].plot(valRange,xTau,c="b",ls="--")
    ax[1].axvline(gateData["th"],color="g",linestyle="--")
    ax[1].set_title(f"{gateName} tau")
    #ax[1].set_ylabel("Time constant (ms)")
    ax[1].set_xlim(gRange)
    
    return xInf

# --- Interactive Plotter Bridge ---
# This was done with Gemini
def update_plot(vHalf, vK, tau0, tau1, th, tk,w_exp,minOpen, gateData, vMin,vMax):
    '''GEMINI'''
    print('vK',vK,'tK',tk)
    vKi = 1.0/vK
    tkInv = 1.0/tk
    valRange = np.linspace(vMin,vMax, 100)
    gateData['vHalf'] = vHalf
    gateData['vKi'] = vKi
    gateData['tau0'] = tau0
    gateData['tau1'] = tau1
    gateData['th'] = th
    gateData['tkInv'] = tkInv
    gateData['exponent'] = w_exp
    gateData['minOpenness'] = minOpen
    # gateData = genGate(vHalf, vKi, tau0, tau1, th, tkInv)

    xInf = sigmoid(valRange, gateData["vHalf"], gateData["vKi"], gateData["minOpenness"])
    xInf = np.power(xInf,w_exp)
    xTau = gateData["tau0"] + gateData["tau1"] * sigmoid(valRange, gateData["th"], gateData["tkInv"], 0)

    fig, ax = plt.subplots(2, 1, figsize=(7, 5), dpi=110, sharex=True)
    
    # Top Plot: x_inf
    ax[0].plot(valRange, xInf, c="b", lw=2)
    ax[0].axvline(gateData["vHalf"], color="g", linestyle="--", alpha=0.7)
    ax[0].set_title("Steady State Openness ($x_{\\infty}$)")
    ax[0].set_ylabel("Open Probability")
    ax[0].set_ylim(-0.02, 1.02)
    ax[0].grid(True, linestyle=":", alpha=0.6)

    # Bottom Plot: tau
    ax[1].plot(valRange, xTau, c="r", lw=2)
    ax[1].axvline(gateData["th"], color="g", linestyle="--", alpha=0.7)
    ax[1].set_title("Time Constant ($\\tau$)")
    ax[1].set_ylabel("Tau (ms)")
    ax[1].set_xlabel("Voltage (mV)")
    ax[1].grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.show()

def createGateUI(gateData,vMin=-60,vMax=40):
    
    # --- Widget UI Layout Construction ---
    style = {'description_width': 'initial'}
    slider_layout = Layout(width='95%')
    print('ui vk',1.0/gateData['vKi'], 'ui tkinv',1.0/gateData['tkInv'])
    w_vHalf = widgets.FloatSlider(value=gateData['vHalf'], min=vMin, max=vMax, step=(vMax-vMin)/100.0, description='V Half (mV)', style=style, layout=slider_layout)
    w_Ki = widgets.FloatSlider(value=1.0/gateData['vKi'], min=-2, max=2, step=0.01, description='V Slope (mV)', style=style, layout=slider_layout)
    w_tau0 = widgets.FloatSlider(value=gateData['tau0'], min=0.1*gateData['tau0'], max=10*gateData['tau0'], step=(gateData['tau0'])/10, description='Base Tau (ms)', style=style, layout=slider_layout)
    w_tau1 = widgets.FloatSlider(value=gateData['tau1'], min=0.0, max=10.0, step=0.1, description='Amp Tau (ms)', style=style, layout=slider_layout)
    w_th = widgets.FloatSlider(value=gateData['th'], min=vMin, max=vMax, step=(vMax-vMin)/100.0, description='Tau Mid (mV)', style=style, layout=slider_layout)
    w_tk = widgets.FloatSlider(value=1.0/gateData['tkInv'],  min=-2, max=2, step=0.01,  description='Tau Slope (mV)', style=style, layout=slider_layout)
    w_exponent = widgets.FloatSlider(value=gateData['exponent'], min=0, max=5, step=0.05, description='Exponent', style=style, layout=slider_layout)
    w_minOpen = widgets.FloatSlider(value=gateData['minOpenness'], min=0, max=1, step=0.05, description='min Open', style=style, layout=slider_layout)

    # Map UI controls directly to plotting parameters
    ui_output = interactive_output(
        update_plot,  
        {'vHalf': w_vHalf, 'vK': w_Ki, 'tau0': w_tau0, 'tau1': w_tau1, 'th': w_th, 'tk': w_tk,'w_exp':w_exponent,'minOpen':w_minOpen, 
        "gateData":fixed(gateData),'vMin':fixed(vMin),'vMax':fixed(vMax)}
    )

    # Arrange controls into clean columns next to the canvas
    left_box = VBox([w_vHalf, w_Ki, w_exponent,w_minOpen], layout=Layout(width='50%'))
    right_box = VBox([w_tau0, w_tau1, w_th, w_tk], layout=Layout(width='50%'))
    controls_panel = HBox([left_box, right_box], layout=Layout(margin='0 0 10px 0'))

    # Render the dashboard
    display(VBox([controls_panel, ui_output]))

# Synapses   
Synapses are a bit of a hack implementation. 
The idea is to immediately change the synapse gate on the post synatpic side from 0 to 1 once a spikeing signal is received. 
A spike is sensed pre-synaptically when voltage is >0 mV, modelling the voltage activated L type calcium channels with a reveral potential near 0.   
Ca influx causes vesicle exoctyosis, releasing neurotransmitter into the synaptic cleft. 
Neurotransmitter binds, opening an ion channel.  

Spikes are detected presynatpically when the voltage crosses 0 mV. This binary detected signal is sent to the delay buffer. 
On the post synaptic side, a neuron detects the spike signal>0.5.  
This detection happens via a very sharp sigmoid function. The spike gate jumps from 0 to 1 once a spike is detected.  
Once the spike is gone, the channel activation decays with a timeconstant equal to xt1.
To guarantee correct dynamics, you need to know the time constant. This is used to set tau0. 

In [ ]:
NaChannel = loadChannel('NaV1.6')
KChannel = loadChannel('genericKv')
leak = loadChannel('leak')

TS = 0.1
xt1 = 5.0 
AMPAsyn = {"name":'AMPA',
            "ion":'Cation',
            "Erev": 0,
            "conductance": 5.0,
            "CaCurrent": 0, #0 if this channel does not produce a change in Ca concentration
            "gates":{
            "vGate1":genOpenGate(),
            "vGate2":genOpenGate(),
            "CaGate":genOpenGate(),
            "spikeGate": genSynapseGate(xt1,TS)}}

GABAsyn = deepcopy(AMPAsyn)
GABAsyn['name'] = 'GABAa'
GABAsyn['Erev'] =  -70.0

channels = {'leak':leak,"genericKv":KChannel,'Nav1.6':NaChannel}
neuronDict = {'test': [ Neuron(0,'test','excitatory',AISchannels={'leak':leak}, AISbias=0.0,
                               neuronTypeToSynapseList={'excitatory':['AMPA','GABAa']} ),
                     Neuron(1,'test','excitatory',AISchannels=channels,AISbias=0)] }

synapses = {"AMPA":AMPAsyn,'GABAa':GABAsyn}
el = [[0,1,0]] #source 0 target 1 segment 0
CTXToSTNSynapses = createSynapses(neuronDict['test'],neuronDict['test'],None,
                                  0,1,TS,synapses,el)

    # GPeToSTNSynapses = createSynapses(GPeSpikers,STNNeurons,None,wGPeToSTN,4,TS,synapses=GSInhSynapses,edgeList=GSEL)

synapseDict = {"CTXToSTN":CTXToSTNSynapses} 


spikerDict = {}
simData = genSimData(neuronDict,spikerDict,synapseDict,TS)

allParams = {'simData': widgets.fixed(simData)}
gateParameters =  [ 
                ('Nav1.6','vGate1','tau0',(0.05,3)),
                ('Nav1.6','vGate1','tau1',(0.05,8)),
                ('AMPA','spikeGate','tau0',(0.05,0.2)),
                ('AMPA','spikeGate','tau1',(0.05,10)),
                ('genericKv','vGate1','tau0',(1,30)),
                ('genericKv','vGate1','tau1',(1,30)),]

genUI_SingleNeuronSim(allParams, gateParameters,TS)

In [ ]:
neuronDict = {
    'AISchannels': ['leak','genericKv','Nav1.6'],
    'neuronType': 'excitatory',
    'neuronToSynapseList': {'excitatory':['AMPA'],'inhibitory':['GABA']},
}

In [ ]:
def saveSingleCompartmentNeuron(neuronDict):
    

In [ ]:
def saveChannel(channelDict, libraryDir = CHANNEL_LIBRARY_PATH, template=CHANNEL_TEMPLATE, overwrite=False):
    if "name" not in channelDict:
        raise ValueError("channelDict must contain a 'name' key")

    validateChannel(channelDict, template)
  
    os.makedirs(libraryDir, exist_ok=True)
    fileName = slugify(channelDict["name"]) + ".json"
    filePath = os.path.join(libraryDir, fileName)

    if os.path.exists(filePath) and not overwrite:
        raise FileExistsError(f"{filePath} already exists (overwrite=False)")

    tmpPath = filePath + ".tmp"
    with open(tmpPath, "w") as f:
        json.dump(channelDict, f, indent=2, allow_nan=False)

    os.replace(tmpPath, filePath)
    return filePath

def loadChannel(name, libraryDir = CHANNEL_LIBRARY_PATH, template=CHANNEL_TEMPLATE):
    fileName = slugify(name) + ".json"
    filePath = os.path.join(libraryDir, fileName)
    if not os.path.exists(filePath):
        raise FileNotFoundError(f"No channel file found at {filePath}")
    with open(filePath, "r") as f:
        data = json.load(f)
    validateChannel(data, template)
    return data